In [3]:
import json
import faiss
import numpy as np
import requests

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from rank_bm25 import BM25Okapi


# =========================================================
# LOAD DOCUMENTS
# =========================================================

with open("mne_docs_test.json", "r") as f:
    documents = json.load(f)

print(f"Loaded {len(documents)} documents.")


# =========================================================
# LOAD EMBEDDINGS + FAISS INDEX
# =========================================================

embeddings = np.load("mne_embeddings.npy")

index = faiss.read_index(
    "mne_faiss.index"
)

print("Embeddings shape:", embeddings.shape)
print("FAISS index loaded.")


# =========================================================
# LOAD MODELS
# =========================================================

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Models loaded.")


# =========================================================
# BUILD BM25 INDEX
# =========================================================

bm25_corpus = []

for doc in documents:

    text = (
        doc["function_name"]
        + " "
        + doc["description"]
    )

    bm25_corpus.append(
        text.lower().split()
    )

bm25 = BM25Okapi(bm25_corpus)

print("BM25 index created.")


# =========================================================
# QUERY
# =========================================================

query = """
Generate constraints and test cases
for mne.io.read_raw_edf
"""


# =========================================================
# QUERY EMBEDDING
# =========================================================

query_embedding = embedding_model.encode(
    [query]
)

query_embedding = np.array(
    query_embedding,
    dtype="float32"
)

print("Query embedding shape:", query_embedding.shape)


# =========================================================
# DENSE RETRIEVAL (FAISS)
# =========================================================

k_dense = 3

distances, dense_indices = index.search(
    query_embedding,
    k_dense
)

dense_results = dense_indices[0].tolist()

print("\nDense Retrieval Results:")
print(dense_results)


# =========================================================
# SPARSE RETRIEVAL (BM25)
# =========================================================

tokenized_query = query.lower().split()

bm25_scores = bm25.get_scores(
    tokenized_query
)

bm25_indices = np.argsort(
    bm25_scores
)[::-1][:1]

bm25_results = bm25_indices.tolist()

print("\nBM25 Retrieval Results:")
print(bm25_results)


# =========================================================
# HYBRID MERGE
# =========================================================

hybrid_indices = list(
    set(
        dense_results + bm25_results
    )
)

print("\nHybrid Retrieval Results:")
print(hybrid_indices)


# =========================================================
# COLLECT HYBRID DOCS
# =========================================================

hybrid_docs = []

for idx in hybrid_indices:
    hybrid_docs.append(documents[idx])

print(f"\nCollected {len(hybrid_docs)} documents.")


# =========================================================
# PREPARE RERANKER PAIRS
# =========================================================

pairs = []

for doc in hybrid_docs:

    combined_text = f"""
    Function:
    {doc['function_name']}

    Description:
    {doc['description']}
    """

    pairs.append(
        (query, combined_text)
    )

print(f"\nPrepared {len(pairs)} reranking pairs.")


# =========================================================
# CROSS-ENCODER RERANKING
# =========================================================

scores = reranker.predict(pairs)

reranked_results = list(
    zip(scores, hybrid_docs)
)

reranked_results = sorted(
    reranked_results,
    key=lambda x: x[0],
    reverse=True
)


# =========================================================
# DISPLAY FINAL TOP RESULT
# =========================================================

top_doc = reranked_results[0]

score, doc = top_doc

print("\n" + "=" * 60)

print(f"Top Score: {score:.4f}")

print("\nFunction:")
print(doc["function_name"])

print("\nDescription:")
print(doc["description"][:1000])


# =========================================================
# BUILD FINAL CONTEXT
# =========================================================

final_context = f"""

Function:
{doc["function_name"]}

Description:
{doc["description"]}

Parameters:
{json.dumps(doc["parameters"], indent=2)}

"""


# =========================================================
# PROMPT CONSTRUCTION
# =========================================================

prompt = f"""
You are an API constraint and test generation system.

Using ONLY the retrieved API documentation below,
generate parameter-level constraints and corresponding test cases.

For each inferred constraint provide:

1. Parameter Name
2. Constraint
3. Short Reasoning
4. Valid Example
5. Invalid pytest-style Test Case

Focus on:
- datatype constraints
- invalid input conditions
- mutually conflicting parameters
- filesystem-related failures
- boundary conditions

Rules:
- ONLY use behaviors supported by the documentation
- Do NOT invent undocumented parameters
- Do NOT assume hidden implementation details
- Keep outputs concise and structured

Retrieved Documentation:
{final_context}
"""


# =========================================================
# OLLAMA GENERATION
# =========================================================

url = "http://localhost:11434/api/generate"

payload = {
    "model": "qwen3:8b-q4_K_M",
    "prompt": prompt,
    "stream": False
}

response = requests.post(
    url,
    json=payload
)

result = response.json()

print("\n" + "=" * 60)
print("GENERATED CONSTRAINTS")
print("=" * 60)

print(result["response"])

Loaded 60 documents.
Embeddings shape: (60, 384)
FAISS index loaded.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Models loaded.
BM25 index created.
Query embedding shape: (1, 384)

Dense Retrieval Results:
[0, 52, 55]

BM25 Retrieval Results:
[0]

Hybrid Retrieval Results:
[0, 52, 55]

Collected 3 documents.

Prepared 3 reranking pairs.

Top Score: -0.4770

Function:
mne.io.read_raw_edf

Description:


GENERATED CONSTRAINTS
### Constraints and Test Cases for `mne.io.read_raw_edf`

---

#### **1. `input_fname`**
**Constraint**: Must be a valid path-like object or file-like object (with `preload=True` if file-like).  
**Reasoning**: The function requires a valid file path or file-like object, with specific handling for file-like objects.  
**Valid Example**: `"example.edf"`  
**Invalid Test Case**:  
```python
pytest.raises(ValueError, mne.io.read_raw_edf, input_fname="invalid_path", preload=False)
```

---

#### **2. `eog`**
**Constraint**: Must be `None`, a list/tuple of channel names/indices.  
**Reasoning**: Invalid types (e.g., strings) or values not matching channel indices/names are unsuppor

In [ ]:
function_name = ""

In [5]:
import json
import faiss
import numpy as np
import requests

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from rank_bm25 import BM25Okapi


# =========================================================
# LOAD DOCUMENTS
# =========================================================

with open("mne_docs_test.json", "r") as f:
    documents = json.load(f)

print(f"Loaded {len(documents)} documents.")


# =========================================================
# LOAD EMBEDDINGS + FAISS INDEX
# =========================================================

embeddings = np.load("mne_embeddings.npy")

index = faiss.read_index(
    "mne_faiss.index"
)

print("Embeddings shape:", embeddings.shape)
print("FAISS index loaded.")


# =========================================================
# LOAD MODELS
# =========================================================

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Models loaded.")


# =========================================================
# BUILD BM25 INDEX
# =========================================================

bm25_corpus = []

for doc in documents:

    text = (
        doc["function_name"]
        + " "
        + doc["description"]
    )

    bm25_corpus.append(
        text.lower().split()
    )

bm25 = BM25Okapi(bm25_corpus)

print("BM25 index created.")


# =========================================================
# QUERY
# =========================================================

target_api = "mne.io.read_raw_edf"

query = f"""
Generate constraints and test cases
for {target_api}
"""


# =========================================================
# QUERY EMBEDDING
# =========================================================

query_embedding = embedding_model.encode(
    [query]
)

query_embedding = np.array(
    query_embedding,
    dtype="float32"
)

print("Query embedding shape:", query_embedding.shape)


# =========================================================
# DENSE RETRIEVAL (FAISS)
# =========================================================

k_dense = 2

distances, dense_indices = index.search(
    query_embedding,
    k_dense
)

dense_results = dense_indices[0].tolist()

print("\nDense Retrieval Results:")
print(dense_results)


# =========================================================
# SPARSE RETRIEVAL (BM25)
# =========================================================

tokenized_query = query.lower().split()

bm25_scores = bm25.get_scores(
    tokenized_query
)

bm25_indices = np.argsort(
    bm25_scores
)[::-1][:1]

bm25_results = bm25_indices.tolist()

print("\nBM25 Retrieval Results:")
print(bm25_results)


# =========================================================
# HYBRID MERGE
# =========================================================

hybrid_indices = list(
    set(
        dense_results + bm25_results
    )
)

print("\nHybrid Retrieval Results:")
print(hybrid_indices)


# =========================================================
# COLLECT HYBRID DOCS
# =========================================================

hybrid_docs = []

for idx in hybrid_indices:
    hybrid_docs.append(documents[idx])

print(f"\nCollected {len(hybrid_docs)} documents.")


# =========================================================
# FUNCTION NAME FILTERING
# =========================================================

filtered_docs = []

for doc in hybrid_docs:

    if target_api in doc["function_name"]:
        filtered_docs.append(doc)

print(f"\nFiltered docs count: {len(filtered_docs)}")


# =========================================================
# PREPARE RERANKER PAIRS
# =========================================================

pairs = []

for doc in filtered_docs:

    combined_text = f"""
    Function:
    {doc['function_name']}

    Parameters:
    {json.dumps(doc['parameters'], indent=2)}
    """

    pairs.append(
        (query, combined_text)
    )

print(f"\nPrepared {len(pairs)} reranking pairs.")


# =========================================================
# CROSS-ENCODER RERANKING
# =========================================================

scores = reranker.predict(pairs)

reranked_results = list(
    zip(scores, filtered_docs)
)

reranked_results = sorted(
    reranked_results,
    key=lambda x: x[0],
    reverse=True
)


# =========================================================
# DISPLAY FINAL TOP RESULT
# =========================================================

top_doc = reranked_results[0]

score, doc = top_doc

print("\n" + "=" * 60)

print(f"Top Score: {score:.4f}")

print("\nFunction:")
print(doc["function_name"])


# =========================================================
# BUILD FINAL CONTEXT
# =========================================================

final_context = f"""

Function:
{doc["function_name"]}

Parameters:
{json.dumps(doc["parameters"], indent=2)}

"""


# =========================================================
# PROMPT CONSTRUCTION
# =========================================================

prompt = f"""
You are an API constraint and test generation system.

Generate constraints ONLY for:
{target_api}

Using ONLY the retrieved API documentation below,
generate parameter-level constraints and corresponding test cases.

For each inferred constraint provide:

1. Parameter Name
2. Constraint
3. Short Reasoning
4. Valid Example
5. Invalid pytest-style Test Case

Focus on:
- datatype constraints
- invalid input conditions
- mutually conflicting parameters
- filesystem-related failures
- boundary conditions

Rules:
- ONLY use behaviors explicitly supported by the documentation
- If a behavior is not explicitly specified,
  say: "Not explicitly specified"
- Do NOT invent undocumented parameters
- Do NOT assume hidden implementation details
- Keep outputs concise and structured

Retrieved Documentation:
{final_context}
"""


# =========================================================
# OLLAMA GENERATION
# =========================================================

url = "http://localhost:11434/api/generate"

payload = {
    "model": "qwen3:8b-q4_K_M",
    "prompt": prompt,
    "stream": False
}

response = requests.post(
    url,
    json=payload
)

result = response.json()

print("\n" + "=" * 60)
print("GENERATED CONSTRAINTS")
print("=" * 60)

print(result["response"])

Loaded 60 documents.
Embeddings shape: (60, 384)
FAISS index loaded.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Models loaded.
BM25 index created.
Query embedding shape: (1, 384)

Dense Retrieval Results:
[0, 52]

BM25 Retrieval Results:
[0]

Hybrid Retrieval Results:
[0, 52]

Collected 2 documents.

Filtered docs count: 1

Prepared 1 reranking pairs.

Top Score: 0.6119

Function:
mne.io.read_raw_edf


KeyboardInterrupt: 